In [1]:
import torch
import os
from torchvision.transforms import v2
from torchvision import datasets
from pathlib import Path
from torch.utils.data import DataLoader
import torchvision.models as models
from tqdm.auto import tqdm
import torchinfo
from timeit import default_timer as timer
from torchinfo import summary

ModuleNotFoundError: No module named 'torchinfo'

In [3]:
train_transform = v2.Compose([
    v2.Resize(size=(224,224)),
    v2.TrivialAugmentWide(num_magnitude_bins=31),
    v2.ToTensor()
])
test_transform = v2.Compose([
    v2.Resize(size=(224,224)),
    v2.ToTensor()
])

c:\Users\Fidel S\AppData\Local\Programs\Python\Python314\Lib\site-packages\torchvision\transforms\v2\_deprecated.py:42: UserWarning: The transform `ToTensor()` is deprecated and will be removed in a future release. Instead, please use `v2.Compose([v2.ToImage(), v2.ToDtype(torch.float32, scale=True)])`.Output is equivalent up to float precision.
  warnings.warn(


In [4]:
test_dir = Path("Data/test")
train_dir = Path("Data/train")

In [9]:
train_data = datasets.ImageFolder(root=train_dir, transform=train_transform, target_transform=None)
test_data = datasets.ImageFolder(root=test_dir, transform=test_transform)

In [10]:
BATCH_SIZE = 32

train_dataloader = DataLoader(dataset=train_data,
                              batch_size=BATCH_SIZE,
                              num_workers=4,
                              shuffle=True)
test_dataloader = DataLoader(dataset=test_data,
                             batch_size=BATCH_SIZE,
                             num_workers=4,
                             shuffle=False,
                             )

In [11]:
weights = models.ResNet18_Weights.DEFAULT
model_1 = models.resnet18(weights=weights)

In [12]:
summary(model=model_1,
        input_size=(1,3,224,224))

Layer (type:depth-idx)                   Output Shape              Param #
ResNet                                   [1, 1000]                 --
├─Conv2d: 1-1                            [1, 64, 112, 112]         9,408
├─BatchNorm2d: 1-2                       [1, 64, 112, 112]         128
├─ReLU: 1-3                              [1, 64, 112, 112]         --
├─MaxPool2d: 1-4                         [1, 64, 56, 56]           --
├─Sequential: 1-5                        [1, 64, 56, 56]           --
│    └─BasicBlock: 2-1                   [1, 64, 56, 56]           --
│    │    └─Conv2d: 3-1                  [1, 64, 56, 56]           36,864
│    │    └─BatchNorm2d: 3-2             [1, 64, 56, 56]           128
│    │    └─ReLU: 3-3                    [1, 64, 56, 56]           --
│    │    └─Conv2d: 3-4                  [1, 64, 56, 56]           36,864
│    │    └─BatchNorm2d: 3-5             [1, 64, 56, 56]           128
│    │    └─ReLU: 3-6                    [1, 64, 56, 56]           --
│

In [13]:
device = "cuda" if torch.cuda.is_available() else "cpu"

In [14]:
def train_step(model: torch.nn.module,
          loss_fn: torch.nn.module,
          optimizer: torch.optim.optimizer,
          dataloader: torch.utils.data.DataLoader,
          device=device,):
    model.train()

    train_loss, train_acc = 0,0

    for batch, (X,y) in enumerate(dataloader):
        X,y = X.to(device), y.to(device)

        y_pred = model(X)

        loss = loss_fn(y_pred, y)
        train_loss += loss

        optimizer.zero_grad()
        loss.backward()
        optimizer.step()

        y_pred_class = torch.argmax(torch.softmax(y_pred, dim=1), dim=1)
        train_acc += (y_pred_class == y).sum().item()/len(y_pred)

    train_loss = train_loss/len(dataloader)
    train_acc = train_acc/len(dataloader)

    return train_loss, train_acc


In [15]:
def test_step(model: torch.nn.module,
         loss_fn: torch.nn.module,
         dataloader: torch.utils.data.DataLoader,
         device=device):
    model.eval()

    test_loss, test_acc = 0,0

    with torch.inference_mode():
        for batch, (X,y) in enumerate(dataloader):
            X,y = X.to(device), y.to(device)

            y_pred = model(X)

            loss = loss_fn(y_pred, y)
            test_loss += loss

            y_pred_class = torch.argmax(torch.softmax(y_pred, dim=1), dim=1)
            test_acc += (y_pred_class == y).sum().item()/len(y_pred)
        
    test_loss = test_loss/len(dataloader)
    test_acc = test_acc/len(dataloader)
    return test_loss, test_acc

In [16]:
def train(model: torch.nn.module,
          loss_fn: torch.nn.module,
          test_dataloader: torch.utils.data.DataLoader,
          train_dataloader: torch.utils.data.DataLoader,
          optimizer: torch.optim.optimizer = torch.nn.CrossEntropyLoss(),
          epochs: int = 5,
          device = device):
    
    results = {"test_loss": [],
               "test_acc": [],
               "train_loss": [],
               "train_acc": []}
    
    for epoch in tqdm(epochs):

        train_loss, train_acc = train_step(model=model_1,
                                           loss_fn=loss_fn,
                                           optimizer=optimizer,
                                           dataloader=train_dataloader,
                                           device=device)
        test_loss, test_acc = test_step(model=model_1,
                                        loss_fn=loss_fn,
                                        dataloader=test_dataloader,
                                        device=device)
        
        print(f"Epoch: {epoch+1} | train_loss: {train_loss:.4f} | train_acc: {train_acc:.4f} | test_loss: {test_loss:.4f} | test_acc: {test_acc:.4f}")

        results["test_loss"].append(test_loss)
        results["test_acc"].append(test_loss)
        results["train_loss"].append(train_loss)
        results["train_acc"].append(train_acc)
    
    return results

In [ ]:
EPOCHS = 1

loss_fn = torch.nn.CrossEntropyLoss()
optimizer = torch.optim.Adam()

In [ ]:
!nvidia-smi